# Descriptive statistics, covariance, correlation and the central limit theorem

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## From probability to statistics
Probability says "if we know the distribution, what will the data look like". Statistics goes the other way: "we have the data; what can we say about the distribution?" Machine learning is the same task.

## Summarizing one column of data
Data: 2, 4, 4, 4, 5, 5, 7, 9

| Statistic | Calculation | Value |
|---|---|---|
| Mean $\bar{x}$ | $40/8$ | 5 |
| Median | Between 4 and 5 | 4.5 |
| Mode | Most frequent | 4 |
| Sum of squared deviations | $9+1+1+1+0+0+4+16$ | 32 |
| Population variance | $32/8$ | 4 |
| Population standard deviation | $\sqrt4$ | 2 |
| Sample variance $s^2$ | $32/7$ | 4.571 |
| Sample standard deviation $s$ | $\sqrt{4.571}$ | 2.138 |

> Why divide by $n-1$? Because the mean was computed from the same data, which uses up one "degree of freedom". In NumPy `np.var` divides by n by default; pass `ddof=1` for a sample. Pandas uses $n-1$ by default.

| Mean | Median |
|---|---|
| Uses every value | Depends only on the order |
| Sensitive to outliers | Robust to outliers |
| For symmetric distributions | For income, house prices and any skewed distribution |

## Covariance: how do two variables move together?
$$\mathrm{cov}(x,y)=\frac{1}{n-1}\sum_i(x_i-\bar{x})(y_i-\bar{y})$$

Example: $x=(1,2,3,4,5)$ and $y=(2,4,5,4,5)$ with $\bar{x}=3$ and $\bar{y}=4$:

| $x_i-\bar{x}$ | $y_i-\bar{y}$ | Product |
|---|---|---|
| −2 | −2 | 4 |
| −1 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 2 | 1 | 2 |

The sum is 6, so $\mathrm{cov}=6/4=1.5$. Positive means that when x goes up, y usually goes up too.

## Correlation: covariance without units
The size of a covariance depends on the units. Dividing by the standard deviations gives a number between −1 and 1:

$$r=\frac{\mathrm{cov}(x,y)}{s_x\,s_y}=\frac{1.5}{1.581\times1.225}=0.775$$

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-stats

> [The correlation coefficient from −1 (a perfect falling line) to 1 (a perfect rising line).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_corr.png)

*The correlation coefficient from −1 (a perfect falling line) to 1 (a perfect rising line).*

| r | Meaning |
|---|---|
| Near 1 | Strong positive linear relation |
| Near 0 | No **linear** relation (there may be a non-linear one!) |
| Near −1 | Strong negative linear relation |

## Two important warnings
> [Three datasets with almost the same correlation and completely different shapes.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_anscombe.png)

*Three datasets with almost the same correlation and completely different shapes.*

- **Always plot the data.** The number r shows neither curvature nor outliers.
- **Correlation is not causation.** Ice-cream sales and drownings are correlated because both depend on hot weather.

## The covariance matrix
For d features, all the covariances fit in a symmetric $d\times d$ matrix. Its diagonal holds the variances:

$$\Sigma=\frac{1}{n-1}X_c^\top X_c=\begin{bmatrix}2.5&1.5\\1.5&1.5\end{bmatrix}$$

where $X_c$ is the centred data. The eigenvectors of this matrix are the principal components of PCA.

## The central limit theorem
The mean of n independent samples from **any** distribution (with finite variance) is approximately normal when n is large:

$$\bar{X}\approx N\!\left(\mu,\ \frac{\sigma^2}{n}\right)$$

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-stats

The standard deviation of the mean is called the **standard error**: $SE=\sigma/\sqrt{n}$.

| n | Standard error when $\sigma=10$ |
|---|---|
| 4 | 5 |
| 25 | 2 |
| 100 | 1 |
| 10000 | 0.1 |

To halve the error you need **four times** the data. This is why accuracy on a small test set is unreliable: with 100 test samples and 90% accuracy the standard error is $\sqrt{0.9\times0.1/100}=0.03$, so the true accuracy could be anywhere between 84% and 96%.



In [ ]:
import numpy as np

d = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print("mean:", d.mean(), " median:", np.median(d))
print("population variance:", d.var(), "  std:", d.std())
print("sample variance    :", round(d.var(ddof=1), 3), "  std:", round(d.std(ddof=1), 3))

# An outlier moves the mean but hardly moves the median
salaries = np.array([30, 32, 35, 36, 40, 41, 45, 900])
print("salaries: mean =", salaries.mean(), "  median =", np.median(salaries))

x = np.array([1.0, 2, 3, 4, 5]); y = np.array([2.0, 4, 5, 4, 5])
cov = np.sum((x - x.mean()) * (y - y.mean())) / (len(x) - 1)
r = cov / (x.std(ddof=1) * y.std(ddof=1))
print("cov =", cov, "  r =", round(r, 4))
print("covariance matrix:"); print(np.cov(x, y))
print("correlation matrix:"); print(np.corrcoef(x, y).round(4))

# Correlation only measures LINEAR relations
t = np.linspace(-3, 3, 101)
print("r between t and t^2:", round(np.corrcoef(t, t ** 2)[0, 1], 4))

# Central limit theorem with a very skewed distribution (exponential: mean 1, std 1)
rng = np.random.default_rng(0)
for n in [1, 4, 25, 100]:
    means = rng.exponential(1.0, size=(20_000, n)).mean(axis=1)
    print(f"n = {n:3d}: std of the means = {means.std():.3f}   sigma / sqrt(n) = {1 / np.sqrt(n):.3f}")

# Standard error of an accuracy measured on a test set
for n_test in [100, 1000, 10_000]:
    se = np.sqrt(0.9 * 0.1 / n_test)
    print(f"test size {n_test:6d}: accuracy 0.90 +/- {2 * se:.3f}")

### Exercise

For $x=(2,4,6,8)$ and $y=(1,3,2,5)$ compute the sample covariance and the correlation coefficient by hand.

In [ ]:
# Your answer here

### Exercise

In the correlation lab set ρ to 0.9, then add one far point in the bottom-right corner. What happens to r?

In [ ]:
# Your answer here